# Classical Machine Learning Models

## Objective

Train and evaluate classical machine-learning models to predict students' final academic outcomes using a provisional feature set intended to represent information available at enrollment.

This notebook will:

1. Load the original dataset.
2. Reproduce the documented feature exclusions.
3. Create a stratified train/test split.
4. Build a preprocessing pipeline.
5. Train logistic regression as the first predictive model.
6. Compare its performance with a simple baseline.

The test set will be held out from model fitting and preprocessing.

In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "data.csv"

df = pd.read_csv(data_path, sep=";")
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Feature Selection

The target is `Target`, containing the three final outcomes: Graduate, Dropout, and Enrolled.

For this initial experiment, we exclude:

- All first- and second-semester features, because they describe information recorded after enrollment.
- `Debtor`, `Tuition fees up to date`, and `Scholarship holder`, because their exact availability at enrollment has not yet been verified.

The three additional exclusions are conservative provisional decisions, not claims that those variables were necessarily recorded after enrollment.

In [4]:
semester_columns = [
    col
    for col in df.columns
    if "1st sem" in col.lower() or "2nd sem" in col.lower()
]

uncertain_timing_columns = [
    "Debtor",
    "Tuition fees up to date",
    "Scholarship holder",
]

excluded_columns = semester_columns + uncertain_timing_columns

candidate_feature_columns = [
    col
    for col in df.columns
    if col != "Target" and col not in excluded_columns
]

X = df[candidate_feature_columns].copy()
y = df["Target"].copy()

print("Semester features excluded:", len(semester_columns))
print("Additional features excluded:", len(uncertain_timing_columns))
print("Selected feature count:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

Semester features excluded: 12
Additional features excluded: 3
Selected feature count: 21
X shape: (4424, 21)
y shape: (4424,)


In [5]:
assert "Target" not in X.columns

assert not any(
    "1st sem" in col.lower() or "2nd sem" in col.lower()
    for col in X.columns
)

assert not any(
    col in X.columns
    for col in uncertain_timing_columns
)

assert X.shape == (4424, 21)
assert len(X) == len(y)

print("Feature-selection checks passed.")

Feature-selection checks passed.


## Train/Test Split

We use an 80/20 stratified split.

The training set is used to fit the model and learn preprocessing parameters. The test set is reserved for evaluating predictions on observations that were not used for fitting.

The fixed random seed makes the split reproducible, and stratification preserves approximately the same target-class proportions in both subsets.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (3539, 21)
X_test: (885, 21)
y_train: (3539,)
y_test: (885,)
